In [ ]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --- 1. 使用 Rasterio 读取 GeoTIFF 文件 ---
# !!! 请将这里的 tif_filename 替换成你的文件路径
tif_filename = '../../data/supplementary_figures/extent/GMW_v3_2020/gmw_v3_2020_0.25d.tif'

with rasterio.open(tif_filename) as src:
    # 读取第一个波段，masked=True 会自动处理无效值
    image_data = src.read(1, masked=True)
    # 将栅格值0也设置为不可见 (NaN)
    # 首先需要将数据类型转为浮点数以支持 NaN
    image_data = image_data.astype('float64')
    image_data[image_data == 0] = np.nan

    # 获取地理范围 (left, right, bottom, top)
    # Rasterio 的 bounds 属性可以直接给出范围，比 gdal 更方便
    img_extent = (src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top)

data1_filename = '../../data/supplementary_figures/obs/ThisStudy/GEB-2020-0221-TabS1_RovaiOnly.xlsx'
data1 = pd.read_excel(data1_filename)
lat1 = data1.iloc[:, 6]
lon1 = data1.iloc[:, 5]

data2_filename = '../../data/supplementary_figures/obs/ThisStudy/Mangrove_MetaData_250311_GEB_ThisStudy_Overlap.xlsx'
data2 = pd.read_excel(data2_filename)
lat2 = data2.iloc[:, 0]
lon2 = data2.iloc[:, 1]

data3_filename = '../../data/supplementary_figures/obs/ThisStudy/Mangrove_MetaData_250311_ThisStudyOnly.xlsx'
data3 = pd.read_excel(data3_filename)
lat3 = data3.iloc[:, 0]
lon3 = data3.iloc[:, 1]

# --- 4. 开始绘图 ---
# 设置字体和字号
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'

# 创建地图
fig, ax = plt.subplots(figsize=(12, 3), dpi=800, subplot_kw={'projection': ccrs.PlateCarree()}, constrained_layout=True)
ax.set_extent([-180, 180, -40, 40], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='lightgray')
ax.add_feature(cfeature.OCEAN, facecolor='white')
# ax.add_feature(cfeature.COASTLINE, linewidth=0.5)

# 绘制红树林范围
green_cmap = ListedColormap(['green'])
mangrove_plot = ax.imshow(image_data, cmap=green_cmap, vmin=0, vmax=1,
                          extent=img_extent, origin='upper', transform=ccrs.PlateCarree())

# 绘制第一个散点数据 (白色)
scatter1 = ax.scatter(lon1, lat1, color='yellow', s=20, edgecolors='k',
                      linewidths=0.5, label='Rovai et al. Only', transform=ccrs.PlateCarree())

# 绘制第二个散点数据 (红色)
scatter2 = ax.scatter(lon2, lat2, color='blue', s=20, edgecolors='k',
                      linewidths=0.5, label='Overlap', transform=ccrs.PlateCarree())

scatter3 = ax.scatter(lon3, lat3, color='red', s=20, edgecolors='k',
                      linewidths=0.5, label='This Study Only', transform=ccrs.PlateCarree())

# 添加图例
# 使用 scatter 对象自动生成图例，这样更灵活
ax.legend(handles=[scatter1, scatter2, scatter3], loc='lower left', frameon=False, markerscale=1.5,
          bbox_to_anchor=(0.01, 0.2), fontsize=14)

# 自定义红树林范围的 "伪" colorbar/图例
cbar_ax = fig.add_axes([0.02, 0.20, 0.04, 0.05])
cbar = plt.colorbar(mangrove_plot, cax=cbar_ax, orientation='horizontal')
cbar.ax.set_xticklabels([])
cbar.ax.tick_params(labelsize=14, length=0)
cbar.outline.set_visible(False)
cbar.ax.text(1.3, 0.5, '2020 mangrove extent', va='center', ha='left', transform=cbar.ax.transAxes, fontsize=14)

# 保存图像
plt.savefig('../../figures/supplementary/figS22_sample_sites.png', dpi=800, bbox_inches='tight')

plt.show()

